In [ ]:
import pandas as pd
import numpy as np
import re
import sqlite3
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/academia_industry_portal")

directories = [
    "data",
    "notebooks",
    "src",
    "database",
    "tests",
    "models",
    "uploads",
    "docs",
]

for directory in directories:
    (PROJECT_DIR / directory).mkdir(
        parents=True, exist_ok=True
    )

print("Project directory:", PROJECT_DIR)
print("Directories created:", directories)

Project directory: /content/academia_industry_portal
Directories created: ['data', 'notebooks', 'src', 'database', 'tests', 'models', 'uploads', 'docs']


In [ ]:
data_dir = PROJECT_DIR / "data"

skills = pd.DataFrame([
    ["SK001", "Python", "Programming", "python programming"],
    ["SK002", "SQL", "Database", "structured query language"],
    ["SK003", "Machine Learning", "AI", "ml"],
    ["SK004", "Pandas", "Data Analysis", "python pandas"],
    ["SK005", "Statistics", "Mathematics", "statistical analysis"],
    ["SK006", "TensorFlow", "Deep Learning", "tf"],
    ["SK007", "Scikit-learn", "Machine Learning", "sklearn"],
    ["SK008", "Power BI", "Visualization", "business intelligence"],
    ["SK009", "Communication", "Soft Skills", "communication skills"],
    ["SK010", "Git", "Development Tools", "version control"],
], columns=[
    "skill_id", "skill_name", "category", "aliases"
])

courses = pd.DataFrame([
    ["C001", "Python for Data Analysis",
     "Python, Pandas, SQL", "Beginner", "University"],
    ["C002", "Machine Learning Fundamentals",
     "Python, Machine Learning, Statistics, Scikit-learn",
     "Intermediate", "University"],
    ["C003", "Deep Learning with TensorFlow",
     "Python, TensorFlow, Machine Learning",
     "Advanced", "University"],
    ["C004", "Business Intelligence",
     "SQL, Power BI, Statistics",
     "Intermediate", "Industry Partner"],
], columns=[
    "course_id", "course_name", "skills_taught",
    "level", "provider"
])

internships = pd.DataFrame([
    ["I001", "Data Analyst Intern", "Example Analytics",
     "Python, SQL, Pandas, Statistics",
     "Analyze datasets and prepare reports", "Remote", True],
    ["I002", "Machine Learning Intern", "Example AI",
     "Python, Machine Learning, Scikit-learn, Statistics",
     "Develop and evaluate predictive models", "Hybrid", True],
    ["I003", "Deep Learning Intern", "Example Vision",
     "Python, TensorFlow, Machine Learning",
     "Experiment with neural network models", "Remote", True],
], columns=[
    "opportunity_id", "title", "company", "required_skills",
    "description", "location", "is_active"
])

jobs = pd.DataFrame([
    ["J001", "Junior Data Scientist", "Example Research",
     "Python, SQL, Machine Learning, Statistics, Pandas",
     "Build data science models and communicate findings",
     "Hybrid", True],
    ["J002", "BI Analyst", "Example Consulting",
     "SQL, Power BI, Statistics, Communication",
     "Create business dashboards and analyze KPIs",
     "On-site", True],
    ["J003", "ML Engineer", "Example Systems",
     "Python, Machine Learning, TensorFlow, Git",
     "Build and maintain machine learning pipelines",
     "Remote", True],
], columns=[
    "opportunity_id", "title", "company", "required_skills",
    "description", "location", "is_active"
])

assessment_questions = pd.DataFrame([
    ["Q001", "Python", "Which type stores key-value pairs?",
     "List", "Dictionary", "Tuple", "Set", "B", "Easy"],
    ["Q002", "SQL", "Which command retrieves rows?",
     "SELECT", "DELETE", "DROP", "UPDATE", "A", "Easy"],
    ["Q003", "Machine Learning",
     "What does a test set estimate?",
     "Training time", "Generalization performance",
     "File size", "CPU speed", "B", "Intermediate"],
], columns=[
    "question_id", "skill_name", "question",
    "option_a", "option_b", "option_c", "option_d",
    "correct_option", "difficulty"
])

datasets = {
    "skills.csv": skills,
    "courses.csv": courses,
    "internships.csv": internships,
    "jobs.csv": jobs,
    "assessment_questions.csv": assessment_questions,
}

for filename, dataframe in datasets.items():
    dataframe.to_csv(data_dir / filename, index=False)

for filename, dataframe in datasets.items():
    print(f"{filename}: {len(dataframe)} rows")

print("\nSaved datasets to:", data_dir)

skills.csv: 10 rows
courses.csv: 4 rows
internships.csv: 3 rows
jobs.csv: 3 rows
assessment_questions.csv: 3 rows

Saved datasets to: /content/academia_industry_portal/data


In [ ]:
import sqlite3

conn = sqlite3.connect(
    "/content/academia_portal.db"
)

conn.execute("""
CREATE TABLE IF NOT EXISTS test_table (
    id INTEGER PRIMARY KEY,
    message TEXT
)
""")

conn.commit()

print("SQLite database created successfully!")

SQLite database created successfully!


In [ ]:

import sqlite3
import pandas as pd

# Connect to your existing database
conn = sqlite3.connect("/content/academia_portal.db")
conn.execute("PRAGMA foreign_keys = ON")

# Create all tables
schema = """
CREATE TABLE IF NOT EXISTS users (
    user_id INTEGER PRIMARY KEY AUTOINCREMENT,
    email TEXT NOT NULL UNIQUE,
    password_hash TEXT NOT NULL,
    role TEXT NOT NULL CHECK (
        role IN ('student', 'faculty', 'company', 'admin')
    ),
    is_active INTEGER NOT NULL DEFAULT 1,
    created_at TEXT DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS student_profiles (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL UNIQUE,
    full_name TEXT NOT NULL,
    degree TEXT,
    specialization TEXT,
    graduation_year INTEGER,
    resume_path TEXT,
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);

CREATE TABLE IF NOT EXISTS skills (
    skill_id INTEGER PRIMARY KEY AUTOINCREMENT,
    skill_name TEXT NOT NULL UNIQUE,
    category TEXT
);

CREATE TABLE IF NOT EXISTS student_skills (
    student_id INTEGER NOT NULL,
    skill_id INTEGER NOT NULL,
    proficiency REAL CHECK (
        proficiency IS NULL OR
        (proficiency >= 0 AND proficiency <= 1)
    ),
    source TEXT DEFAULT 'self_reported',
    PRIMARY KEY (student_id, skill_id),
    FOREIGN KEY (student_id)
        REFERENCES student_profiles(student_id),
    FOREIGN KEY (skill_id) REFERENCES skills(skill_id)
);

CREATE TABLE IF NOT EXISTS companies (
    company_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL UNIQUE,
    company_name TEXT NOT NULL,
    industry TEXT,
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);

CREATE TABLE IF NOT EXISTS opportunities (
    opportunity_id INTEGER PRIMARY KEY AUTOINCREMENT,
    company_id INTEGER NOT NULL,
    title TEXT NOT NULL,
    opportunity_type TEXT NOT NULL CHECK (
        opportunity_type IN ('job', 'internship')
    ),
    description TEXT,
    location TEXT,
    is_active INTEGER NOT NULL DEFAULT 1,
    created_at TEXT DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (company_id) REFERENCES companies(company_id)
);

CREATE TABLE IF NOT EXISTS opportunity_skills (
    opportunity_id INTEGER NOT NULL,
    skill_id INTEGER NOT NULL,
    importance REAL DEFAULT 1.0 CHECK (
        importance >= 0 AND importance <= 1
    ),
    PRIMARY KEY (opportunity_id, skill_id),
    FOREIGN KEY (opportunity_id)
        REFERENCES opportunities(opportunity_id),
    FOREIGN KEY (skill_id) REFERENCES skills(skill_id)
);

CREATE TABLE IF NOT EXISTS applications (
    application_id INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id INTEGER NOT NULL,
    opportunity_id INTEGER NOT NULL,
    status TEXT NOT NULL DEFAULT 'applied' CHECK (
        status IN (
            'applied', 'shortlisted', 'rejected',
            'offered', 'withdrawn'
        )
    ),
    applied_at TEXT DEFAULT CURRENT_TIMESTAMP,
    UNIQUE (student_id, opportunity_id),
    FOREIGN KEY (student_id)
        REFERENCES student_profiles(student_id),
    FOREIGN KEY (opportunity_id)
        REFERENCES opportunities(opportunity_id)
);
"""

conn.executescript(schema)
conn.commit()

print("All database tables created successfully!")

# Display the tables
tables = pd.read_sql_query("""
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
""", conn)

display(tables)


All database tables created successfully!


,table_name
0,applications
1,companies
2,opportunities
3,opportunity_skills
4,skills
5,student_profiles
6,student_skills
7,test_table
8,users


In [ ]:

import sqlite3
import pandas as pd
from pathlib import Path

# Paths
project_dir = Path("/content/academia_industry_portal")
data_dir = project_dir / "data"
db_path = Path("/content/academia_portal.db")

# Connect to database
conn = sqlite3.connect(str(db_path))
conn.execute("PRAGMA foreign_keys = ON")

# Load skills dataset
skills_file = data_dir / "skills.csv"

if skills_file.exists():
    skills_df = pd.read_csv(skills_file)
    skills_df.columns = skills_df.columns.str.strip().str.lower()

    print("Skills CSV columns:", skills_df.columns.tolist())

    # Insert skills into the database
    if "skill_name" not in skills_df.columns:
        raise ValueError(
            "skills.csv must contain a 'skill_name' column."
        )

    if "category" not in skills_df.columns:
        skills_df["category"] = "General"

    for _, row in skills_df.iterrows():
        conn.execute(
            """
            INSERT OR IGNORE INTO skills (skill_name, category)
            VALUES (?, ?)
            """,
            (
                str(row["skill_name"]).strip(),
                str(row["category"]).strip()
                if pd.notna(row["category"]) else "General"
            )
        )

    conn.commit()

    print("\nSkills inserted successfully!")
    display(pd.read_sql_query(
        "SELECT * FROM skills ORDER BY skill_id",
        conn
    ))
else:
    print("skills.csv was not found at:", skills_file)
    print("Check that you previously created the project CSV files.")


Skills CSV columns: ['skill_id', 'skill_name', 'category', 'aliases']

Skills inserted successfully!


,skill_id,skill_name,category
0,1,Python,Programming
1,2,SQL,Database
2,3,Machine Learning,AI
3,4,Pandas,Data Analysis
4,5,Statistics,Mathematics
5,6,TensorFlow,Deep Learning
6,7,Scikit-learn,Machine Learning
7,8,Power BI,Visualization
8,9,Communication,Soft Skills
9,10,Git,Development Tools


In [ ]:

from pathlib import Path
import pandas as pd

data_dir = Path("/content/academia_industry_portal/data")

for filename in [
    "courses.csv",
    "internships.csv",
    "jobs.csv",
    "assessment_questions.csv"
]:
    filepath = data_dir / filename

    print(f"\n--- {filename} ---")

    if filepath.exists():
        df = pd.read_csv(filepath)
        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        display(df.head(3))
    else:
        print("File not found.")



--- courses.csv ---
Rows: 4
Columns: ['course_id', 'course_name', 'skills_taught', 'level', 'provider']


,course_id,course_name,skills_taught,level,provider
0,C001,Python for Data Analysis,"Python, Pandas, SQL",Beginner,University
1,C002,Machine Learning Fundamentals,"Python, Machine Learning, Statistics, Scikit-l...",Intermediate,University
2,C003,Deep Learning with TensorFlow,"Python, TensorFlow, Machine Learning",Advanced,University



--- internships.csv ---
Rows: 3
Columns: ['opportunity_id', 'title', 'company', 'required_skills', 'description', 'location', 'is_active']


,opportunity_id,title,company,required_skills,description,location,is_active
0,I001,Data Analyst Intern,Example Analytics,"Python, SQL, Pandas, Statistics",Analyze datasets and prepare reports,Remote,True
1,I002,Machine Learning Intern,Example AI,"Python, Machine Learning, Scikit-learn, Statis...",Develop and evaluate predictive models,Hybrid,True
2,I003,Deep Learning Intern,Example Vision,"Python, TensorFlow, Machine Learning",Experiment with neural network models,Remote,True



--- jobs.csv ---
Rows: 3
Columns: ['opportunity_id', 'title', 'company', 'required_skills', 'description', 'location', 'is_active']


,opportunity_id,title,company,required_skills,description,location,is_active
0,J001,Junior Data Scientist,Example Research,"Python, SQL, Machine Learning, Statistics, Pandas",Build data science models and communicate find...,Hybrid,True
1,J002,BI Analyst,Example Consulting,"SQL, Power BI, Statistics, Communication",Create business dashboards and analyze KPIs,On-site,True
2,J003,ML Engineer,Example Systems,"Python, Machine Learning, TensorFlow, Git",Build and maintain machine learning pipelines,Remote,True



--- assessment_questions.csv ---
Rows: 3
Columns: ['question_id', 'skill_name', 'question', 'option_a', 'option_b', 'option_c', 'option_d', 'correct_option', 'difficulty']


,question_id,skill_name,question,option_a,option_b,option_c,option_d,correct_option,difficulty
0,Q001,Python,Which type stores key-value pairs?,List,Dictionary,Tuple,Set,B,Easy
1,Q002,SQL,Which command retrieves rows?,SELECT,DELETE,DROP,UPDATE,A,Easy
2,Q003,Machine Learning,What does a test set estimate?,Training time,Generalization performance,File size,CPU speed,B,Intermediate


In [ ]:
# Paths
data_dir = Path("/content/academia_industry_portal/data")
db_path = "/content/academia_portal.db"

# Connect to database
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON")

# Load CSV files
internships = pd.read_csv(data_dir / "internships.csv")
jobs = pd.read_csv(data_dir / "jobs.csv")

# Combine both datasets
internships["opportunity_type"] = "internship"
jobs["opportunity_type"] = "job"
opportunities_df = pd.concat(
    [internships, jobs], ignore_index=True
)

# Normalize column names
opportunities_df.columns = (
    opportunities_df.columns.str.strip().str.lower()
)

required_columns = {
    "title", "company", "required_skills",
    "description", "location", "is_active"
}
missing = required_columns - set(opportunities_df.columns)

if missing:
    raise ValueError(
        f"Missing CSV columns: {sorted(missing)}. "
        f"Available columns: {opportunities_df.columns.tolist()}"
    )

# Build a lookup of existing skills
skill_rows = conn.execute(
    "SELECT skill_id, skill_name FROM skills"
).fetchall()

skill_lookup = {
    name.strip().lower(): skill_id
    for skill_id, name in skill_rows
}

def parse_skill_list(value):
    if pd.isna(value):
        return []
    return [
        item.strip()
        for item in str(value).split(",")
        if item.strip()
    ]

def is_active_value(value):
    return str(value).strip().lower() in {
        "true", "1", "yes", "active"
    }

imported_opportunities = 0
linked_skills = 0
unmatched_skills = set()

for _, row in opportunities_df.iterrows():
    company_name = str(row["company"]).strip()
    title = str(row["title"]).strip()

    if not company_name or not title:
        continue

    # Reuse an existing company if already imported
    company_row = conn.execute(
        "SELECT company_id FROM companies WHERE company_name = ?",
        (company_name,)
    ).fetchone()

    if company_row:
        company_id = company_row[0]
    else:
        # Create a demonstration company account.
        # This placeholder is NOT a usable login credential.
        base = re.sub(
            r"[^a-z0-9]+", ".",
            company_name.lower()
        ).strip(".") or "company"

        email = f"{base}@example.com"
        suffix = 1

        while conn.execute(
            "SELECT 1 FROM users WHERE email = ?", (email,)
        ).fetchone():
            email = f"{base}.{suffix}@example.com"
            suffix += 1

        cursor = conn.execute(
            """
            INSERT INTO users (email, password_hash, role)
            VALUES (?, ?, 'company')
            """,
            (email, "DEMO_ONLY_NOT_A_REAL_PASSWORD_HASH")
        )
        user_id = cursor.lastrowid

        cursor = conn.execute(
            """
            INSERT INTO companies (user_id, company_name)
            VALUES (?, ?)
            """,
            (user_id, company_name)
        )
        company_id = cursor.lastrowid

    opportunity_type = row["opportunity_type"]
    description = (
        "" if pd.isna(row["description"])
        else str(row["description"])
    )
    location = (
        "" if pd.isna(row["location"])
        else str(row["location"])
    )
    active = int(is_active_value(row["is_active"]))

    # Prevent duplicate opportunity records
    existing = conn.execute(
        """
        SELECT opportunity_id
        FROM opportunities
        WHERE company_id = ?
          AND title = ?
          AND opportunity_type = ?
        """,
        (company_id, title, opportunity_type)
    ).fetchone()

    if existing:
        opportunity_id = existing[0]
    else:
        cursor = conn.execute(
            """
            INSERT INTO opportunities (
                company_id, title, opportunity_type,
                description, location, is_active
            )
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                company_id, title, opportunity_type,
                description, location, active
            )
        )
        opportunity_id = cursor.lastrowid
        imported_opportunities += 1

    # Connect opportunity to its required skills
    for skill in parse_skill_list(row["required_skills"]):
        skill_id = skill_lookup.get(skill.lower())

        if skill_id is None:
            unmatched_skills.add(skill)
            continue

        conn.execute(
            """
            INSERT OR IGNORE INTO opportunity_skills
                (opportunity_id, skill_id, importance)
            VALUES (?, ?, 1.0)
            """,
            (opportunity_id, skill_id)
        )

        linked_skills += conn.execute(
            "SELECT changes()"
        ).fetchone()[0]

conn.commit()

print("Import completed!")
print("New opportunities added:", imported_opportunities)
print("Skill links added:", linked_skills)
print("Required skills not found in skills table:",
      sorted(unmatched_skills))

print("\nOpportunities in database:")
display(pd.read_sql_query(
    """
    SELECT
        o.opportunity_id,
        c.company_name,
        o.title,
        o.opportunity_type,
        o.location,
        o.is_active
    FROM opportunities o
    JOIN companies c ON o.company_id = c.company_id
    ORDER BY o.opportunity_type, o.title
    """,
    conn
))


Import completed!
New opportunities added: 0
Skill links added: 0
Required skills not found in skills table: []

Opportunities in database:


,opportunity_id,company_name,title,opportunity_type,location,is_active
0,1,Example Analytics,Data Analyst Intern,internship,Remote,1
1,3,Example Vision,Deep Learning Intern,internship,Remote,1
2,2,Example AI,Machine Learning Intern,internship,Hybrid,1
3,5,Example Consulting,BI Analyst,job,On-site,1
4,4,Example Research,Junior Data Scientist,job,Hybrid,1
5,6,Example Systems,ML Engineer,job,Remote,1


In [ ]:

query = """
SELECT
    o.title AS opportunity,
    c.company_name AS company,
    GROUP_CONCAT(s.skill_name, ', ') AS required_skills
FROM opportunities o
JOIN companies c
    ON o.company_id = c.company_id
LEFT JOIN opportunity_skills os
    ON o.opportunity_id = os.opportunity_id
LEFT JOIN skills s
    ON os.skill_id = s.skill_id
GROUP BY o.opportunity_id, o.title, c.company_name
ORDER BY o.title
"""

display(pd.read_sql_query(query, conn))


,opportunity,company,required_skills
0,BI Analyst,Example Consulting,"SQL, Statistics, Power BI, Communication"
1,Data Analyst Intern,Example Analytics,"Python, SQL, Pandas, Statistics"
2,Deep Learning Intern,Example Vision,"Python, Machine Learning, TensorFlow"
3,Junior Data Scientist,Example Research,"Python, SQL, Machine Learning, Pandas, Statistics"
4,ML Engineer,Example Systems,"Python, Machine Learning, TensorFlow, Git"
5,Machine Learning Intern,Example AI,"Python, Machine Learning, Statistics, Scikit-l..."


In [ ]:

import pandas as pd
import sqlite3

conn = sqlite3.connect("/content/academia_portal.db")

checks = {
    "Companies": "SELECT COUNT(*) FROM companies",
    "Opportunities": "SELECT COUNT(*) FROM opportunities",
    "Skills": "SELECT COUNT(*) FROM skills",
    "Opportunity-skill links": "SELECT COUNT(*) FROM opportunity_skills"
}

for label, query in checks.items():
    count = pd.read_sql_query(query, conn).iloc[0, 0]
    print(f"{label}: {count}")

print("\nSample opportunity-skill mappings:")
display(pd.read_sql_query("""
    SELECT o.title, s.skill_name
    FROM opportunities o
    LEFT JOIN opportunity_skills os
        ON o.opportunity_id = os.opportunity_id
    LEFT JOIN skills s ON os.skill_id = s.skill_id
    ORDER BY o.title
    LIMIT 20
""", conn))


Companies: 6
Opportunities: 6
Skills: 10
Opportunity-skill links: 24

Sample opportunity-skill mappings:


,title,skill_name
0,BI Analyst,SQL
1,BI Analyst,Statistics
2,BI Analyst,Power BI
3,BI Analyst,Communication
4,Data Analyst Intern,Python
5,Data Analyst Intern,SQL
6,Data Analyst Intern,Pandas
7,Data Analyst Intern,Statistics
8,Deep Learning Intern,Python
9,Deep Learning Intern,Machine Learning


In [ ]:

import pandas as pd

def recommend_from_database(student_skills, top_k=5):
    conn = sqlite3.connect("/content/academia_portal.db")

    if not student_skills:
        conn.close()
        raise ValueError("Enter at least one student skill.")

    student_set = {
        str(skill).strip().lower()
        for skill in student_skills
        if str(skill).strip()
    }

    rows = pd.read_sql_query("""
        SELECT
            o.opportunity_id,
            o.title,
            o.opportunity_type,
            o.location,
            c.company_name,
            s.skill_name
        FROM opportunities o
        JOIN companies c
            ON c.company_id = o.company_id
        LEFT JOIN opportunity_skills os
            ON os.opportunity_id = o.opportunity_id
        LEFT JOIN skills s
            ON s.skill_id = os.skill_id
        WHERE o.is_active = 1
        ORDER BY o.opportunity_id
    """, conn)

    conn.close()

    if rows.empty:
        print("No active opportunities found.")
        return pd.DataFrame()

    results = []

    for opportunity_id, group in rows.groupby("opportunity_id"):
        first = group.iloc[0]

        required = {
            str(skill).strip().lower()
            for skill in group["skill_name"].dropna()
            if str(skill).strip()
        }

        matched = student_set & required
        missing = required - student_set

        # No requirements means the match cannot be assessed.
        if not required:
            score = 0.0
            coverage = 0.0
        else:
            coverage = len(matched) / len(required)
            score = coverage

        results.append({
            "title": first["title"],
            "company": first["company_name"],
            "type": first["opportunity_type"],
            "location": first["location"],
            "match_score_percent": round(score * 100, 2),
            "matched_skills": ", ".join(sorted(matched)) or "None",
            "missing_skills": ", ".join(sorted(missing)) or "None",
            "requirements_available": bool(required)
        })

    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values(
        ["requirements_available", "match_score_percent"],
        ascending=[False, False]
    )

    return result_df.head(top_k).reset_index(drop=True)


In [ ]:

student_skills = ["Python", "SQL", "Pandas"]

recommendations = recommend_from_database(
    student_skills=student_skills,
    top_k=5
)

display(recommendations)


,title,company,type,location,match_score_percent,matched_skills,missing_skills,requirements_available
0,Data Analyst Intern,Example Analytics,internship,Remote,75.00,"pandas, python, sql",statistics,True
1,Junior Data Scientist,Example Research,job,Hybrid,60.00,"pandas, python, sql","machine learning, statistics",True
2,Deep Learning Intern,Example Vision,internship,Remote,33.33,python,"machine learning, tensorflow",True
3,Machine Learning Intern,Example AI,internship,Hybrid,25.00,python,"machine learning, scikit-learn, statistics",True
4,BI Analyst,Example Consulting,job,On-site,25.00,sql,"communication, power bi, statistics",True


In [ ]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("/content/academia_portal.db")

print("1. All opportunities:")
display(pd.read_sql_query(
    "SELECT * FROM opportunities", conn
))

print("2. Opportunity counts by active status:")
display(pd.read_sql_query("""
    SELECT is_active, COUNT(*) AS total
    FROM opportunities
    GROUP BY is_active
""", conn))

print("3. Companies:")
display(pd.read_sql_query(
    "SELECT * FROM companies", conn
))

print("4. Imported CSV files:")
from pathlib import Path

data_dir = Path("/content/academia_industry_portal/data")
for filename in ["internships.csv", "jobs.csv"]:
    path = data_dir / filename
    print(filename, "exists:", path.exists())
    if path.exists():
        print("Rows:", len(pd.read_csv(path)))

conn.close()


1. All opportunities:


,opportunity_id,company_id,title,opportunity_type,description,location,is_active,created_at
0,1,1,Data Analyst Intern,internship,Analyze datasets and prepare reports,Remote,1,2026-10-31 09:43:52
1,2,2,Machine Learning Intern,internship,Develop and evaluate predictive models,Hybrid,1,2026-10-31 09:43:52
2,3,3,Deep Learning Intern,internship,Experiment with neural network models,Remote,1,2026-10-31 09:43:52
3,4,4,Junior Data Scientist,job,Build data science models and communicate find...,Hybrid,1,2026-10-31 09:43:52
4,5,5,BI Analyst,job,Create business dashboards and analyze KPIs,On-site,1,2026-10-31 09:43:52
5,6,6,ML Engineer,job,Build and maintain machine learning pipelines,Remote,1,2026-10-31 09:43:52


2. Opportunity counts by active status:


,is_active,total
0,1,6


3. Companies:


,company_id,user_id,company_name,industry
0,1,1,Example Analytics,None
1,2,2,Example AI,None
2,3,3,Example Vision,None
3,4,4,Example Research,None
4,5,5,Example Consulting,None
5,6,6,Example Systems,None


4. Imported CSV files:
internships.csv exists: True
Rows: 3
jobs.csv exists: True
Rows: 3


In [ ]:

import sqlite3
import pandas as pd
from pathlib import Path

# Connect to the database used by your recommendation function
db_path = "/content/academia_portal.db"
data_dir = Path("/content/academia_industry_portal/data")

conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON")

# Load the CSV files
internships = pd.read_csv(data_dir / "internships.csv")
jobs = pd.read_csv(data_dir / "jobs.csv")

internships.columns = internships.columns.str.strip().str.lower()
jobs.columns = jobs.columns.str.strip().str.lower()

internships["opportunity_type"] = "internship"
jobs["opportunity_type"] = "job"

all_data = pd.concat([internships, jobs], ignore_index=True)

# Check required columns before importing
required_columns = {
    "title", "company", "required_skills",
    "description", "location", "is_active"
}
missing = required_columns - set(all_data.columns)

if missing:
    raise ValueError(
        f"Missing columns: {sorted(missing)}. "
        f"Found columns: {all_data.columns.tolist()}"
    )

# Insert companies, opportunities, and their skills
for _, row in all_data.iterrows():
    company_name = str(row["company"]).strip()
    title = str(row["title"]).strip()

    if not company_name or not title:
        continue

    # Create a company record if it doesn't exist
    company = conn.execute(
        "SELECT company_id FROM companies WHERE company_name = ?",
        (company_name,)
    ).fetchone()

    if company:
        company_id = company[0]
    else:
        email_base = "".join(
            ch.lower() if ch.isalnum() else "."
            for ch in company_name
        ).strip(".") or "company"

        email = email_base + "@example.com"
        suffix = 1

        while conn.execute(
            "SELECT 1 FROM users WHERE email = ?", (email,)
        ).fetchone():
            email = f"{email_base}.{suffix}@example.com"
            suffix += 1

        user_id = conn.execute(
            """
            INSERT INTO users (email, password_hash, role)
            VALUES (?, ?, 'company')
            """,
            (email, "DEMO_ONLY_NOT_A_REAL_PASSWORD_HASH")
        ).lastrowid

        company_id = conn.execute(
            """
            INSERT INTO companies (user_id, company_name)
            VALUES (?, ?)
            """,
            (user_id, company_name)
        ).lastrowid

    # Determine whether the opportunity is active
    active = int(
        str(row["is_active"]).strip().lower()
        in {"true", "1", "yes", "active"}
    )

    description = (
        "" if pd.isna(row["description"])
        else str(row["description"])
    )
    location = (
        "" if pd.isna(row["location"])
        else str(row["location"])
    )

    # Insert the opportunity only if it doesn't already exist
    existing = conn.execute(
        """
        SELECT opportunity_id FROM opportunities
        WHERE company_id = ?
          AND title = ?
          AND opportunity_type = ?
        """,
        (company_id, title, row["opportunity_type"])
    ).fetchone()

    if existing:
        opportunity_id = existing[0]
    else:
        opportunity_id = conn.execute(
            """
            INSERT INTO opportunities (
                company_id, title, opportunity_type,
                description, location, is_active
            )
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                company_id, title, row["opportunity_type"],
                description, location, active
            )
        ).lastrowid

    # Link the opportunity to skills already in the skills table
    skill_names = [
        s.strip()
        for s in str(row["required_skills"]).split(",")
        if s.strip() and str(row["required_skills"]) != "nan"
    ]

    for skill_name in skill_names:
        skill = conn.execute(
            "SELECT skill_id FROM skills WHERE LOWER(skill_name) = LOWER(?)",
            (skill_name,)
        ).fetchone()

        # Add a missing skill to the catalogue
        if skill:
            skill_id = skill[0]
        else:
            conn.execute(
                """
                INSERT OR IGNORE INTO skills (skill_name, category)
                VALUES (?, ?)
                """,
                (skill_name, "Imported")
            )
            skill = conn.execute(
                "SELECT skill_id FROM skills WHERE LOWER(skill_name) = LOWER(?)",
                (skill_name,)
            ).fetchone()
            skill_id = skill[0]

        conn.execute(
            """
            INSERT OR IGNORE INTO opportunity_skills
                (opportunity_id, skill_id, importance)
            VALUES (?, ?, 1.0)
            """,
            (opportunity_id, skill_id)
        )

conn.commit()

# Verify the repair
print("Companies:", conn.execute(
    "SELECT COUNT(*) FROM companies"
).fetchone()[0])

print("Opportunities:", conn.execute(
    "SELECT COUNT(*) FROM opportunities"
).fetchone()[0])

print("Opportunity-skill links:", conn.execute(
    "SELECT COUNT(*) FROM opportunity_skills"
).fetchone()[0])

display(pd.read_sql_query(
    """
    SELECT o.opportunity_id, c.company_name, o.title,
           o.opportunity_type, o.is_active
    FROM opportunities o
    JOIN companies c ON c.company_id = o.company_id
    ORDER BY o.opportunity_type, o.title
    """,
    conn
))


Companies: 6
Opportunities: 6
Opportunity-skill links: 24


,opportunity_id,company_name,title,opportunity_type,is_active
0,1,Example Analytics,Data Analyst Intern,internship,1
1,3,Example Vision,Deep Learning Intern,internship,1
2,2,Example AI,Machine Learning Intern,internship,1
3,5,Example Consulting,BI Analyst,job,1
4,4,Example Research,Junior Data Scientist,job,1
5,6,Example Systems,ML Engineer,job,1


In [ ]:

recommendations = recommend_from_database(
    student_skills=["Python", "SQL", "Pandas"],
    top_k=5
)

display(recommendations)


,title,company,type,location,match_score_percent,matched_skills,missing_skills,requirements_available
0,Data Analyst Intern,Example Analytics,internship,Remote,75.00,"pandas, python, sql",statistics,True
1,Junior Data Scientist,Example Research,job,Hybrid,60.00,"pandas, python, sql","machine learning, statistics",True
2,Deep Learning Intern,Example Vision,internship,Remote,33.33,python,"machine learning, tensorflow",True
3,Machine Learning Intern,Example AI,internship,Hybrid,25.00,python,"machine learning, scikit-learn, statistics",True
4,BI Analyst,Example Consulting,job,On-site,25.00,sql,"communication, power bi, statistics",True


In [ ]:

import sqlite3
import pandas as pd
import re
from pathlib import Path

db_path = "/content/academia_portal.db"
data_dir = Path("/content/academia_industry_portal/data")

conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON")

# Read both CSV files
internships = pd.read_csv(data_dir / "internships.csv")
jobs = pd.read_csv(data_dir / "jobs.csv")

internships.columns = internships.columns.str.strip().str.lower()
jobs.columns = jobs.columns.str.strip().str.lower()

internships["opportunity_type"] = "internship"
jobs["opportunity_type"] = "job"
all_data = pd.concat([internships, jobs], ignore_index=True)

# Inspect the source values first
print("Required skills in the source CSV files:")
display(all_data[["title", "required_skills"]])

def split_skills(value):
    if pd.isna(value):
        return []
    return [
        skill.strip()
        for skill in re.split(r"[,;|]", str(value))
        if skill.strip()
    ]

linked_count = 0
unmatched_opportunities = []

for _, row in all_data.iterrows():
    company_name = str(row["company"]).strip()
    title = str(row["title"]).strip()
    kind = row["opportunity_type"]

    company = conn.execute(
        "SELECT company_id FROM companies WHERE company_name = ?",
        (company_name,)
    ).fetchone()

    if not company:
        print("Company not found:", company_name)
        continue

    opportunity = conn.execute(
        """
        SELECT opportunity_id FROM opportunities
        WHERE company_id = ? AND title = ?
          AND opportunity_type = ?
        """,
        (company[0], title, kind)
    ).fetchone()

    if not opportunity:
        print("Opportunity not found:", title)
        continue

    opportunity_id = opportunity[0]
    skills = split_skills(row["required_skills"])

    if not skills:
        unmatched_opportunities.append(title)
        continue

    for skill_name in skills:
        conn.execute(
            """
            INSERT OR IGNORE INTO skills (skill_name, category)
            VALUES (?, 'Imported')
            """,
            (skill_name,)
        )

        skill = conn.execute(
            """
            SELECT skill_id FROM skills
            WHERE LOWER(skill_name) = LOWER(?)
            """,
            (skill_name,)
        ).fetchone()

        if skill:
            result = conn.execute(
                """
                INSERT OR IGNORE INTO opportunity_skills
                    (opportunity_id, skill_id, importance)
                VALUES (?, ?, 1.0)
                """,
                (opportunity_id, skill[0])
            )
            linked_count += result.rowcount

conn.commit()

print("\nNew skill links added:", linked_count)
print("Opportunities with blank required_skills:",
      unmatched_opportunities)

# Verify the final mappings
display(pd.read_sql_query("""
    SELECT
        o.title,
        o.opportunity_type,
        GROUP_CONCAT(s.skill_name, ', ') AS required_skills
    FROM opportunities o
    LEFT JOIN opportunity_skills os
        ON os.opportunity_id = o.opportunity_id
    LEFT JOIN skills s ON s.skill_id = os.skill_id
    GROUP BY o.opportunity_id
    ORDER BY o.title
""", conn))

conn.close()


Required skills in the source CSV files:


,title,required_skills
0,Data Analyst Intern,"Python, SQL, Pandas, Statistics"
1,Machine Learning Intern,"Python, Machine Learning, Scikit-learn, Statis..."
2,Deep Learning Intern,"Python, TensorFlow, Machine Learning"
3,Junior Data Scientist,"Python, SQL, Machine Learning, Statistics, Pandas"
4,BI Analyst,"SQL, Power BI, Statistics, Communication"
5,ML Engineer,"Python, Machine Learning, TensorFlow, Git"



New skill links added: 0
Opportunities with blank required_skills: []


,title,opportunity_type,required_skills
0,BI Analyst,job,"SQL, Statistics, Power BI, Communication"
1,Data Analyst Intern,internship,"Python, SQL, Pandas, Statistics"
2,Deep Learning Intern,internship,"Python, Machine Learning, TensorFlow"
3,Junior Data Scientist,job,"Python, SQL, Machine Learning, Pandas, Statistics"
4,ML Engineer,job,"Python, Machine Learning, TensorFlow, Git"
5,Machine Learning Intern,internship,"Python, Machine Learning, Statistics, Scikit-l..."


In [ ]:

recommendations = recommend_from_database(
    student_skills=["Python", "SQL", "Pandas"],
    top_k=5
)

display(recommendations)


,title,company,type,location,match_score_percent,matched_skills,missing_skills,requirements_available
0,Data Analyst Intern,Example Analytics,internship,Remote,75.00,"pandas, python, sql",statistics,True
1,Junior Data Scientist,Example Research,job,Hybrid,60.00,"pandas, python, sql","machine learning, statistics",True
2,Deep Learning Intern,Example Vision,internship,Remote,33.33,python,"machine learning, tensorflow",True
3,Machine Learning Intern,Example AI,internship,Hybrid,25.00,python,"machine learning, scikit-learn, statistics",True
4,BI Analyst,Example Consulting,job,On-site,25.00,sql,"communication, power bi, statistics",True


In [ ]:

recommendations = recommend_from_database(
    student_skills=["Python", "SQL", "Pandas"],
    top_k=6
)

display(recommendations)


,title,company,type,location,match_score_percent,matched_skills,missing_skills,requirements_available
0,Data Analyst Intern,Example Analytics,internship,Remote,75.00,"pandas, python, sql",statistics,True
1,Junior Data Scientist,Example Research,job,Hybrid,60.00,"pandas, python, sql","machine learning, statistics",True
2,Deep Learning Intern,Example Vision,internship,Remote,33.33,python,"machine learning, tensorflow",True
3,Machine Learning Intern,Example AI,internship,Hybrid,25.00,python,"machine learning, scikit-learn, statistics",True
4,BI Analyst,Example Consulting,job,On-site,25.00,sql,"communication, power bi, statistics",True
5,ML Engineer,Example Systems,job,Remote,25.00,python,"git, machine learning, tensorflow",True


In [ ]:

import sqlite3
import pandas as pd
import re

def normalize_skill(skill):
    skill = str(skill).strip().lower()
    skill = re.sub(r"[\s_-]+", " ", skill)
    aliases = {
        "sklearn": "scikit learn",
        "scikit learn": "scikit learn",
        "ml": "machine learning",
        "powerbi": "power bi"
    }
    return aliases.get(skill, skill)

def recommend_from_database(student_skills, top_k=5):
    if not student_skills:
        raise ValueError("Enter at least one student skill.")

    student_set = {
        normalize_skill(s) for s in student_skills
        if str(s).strip()
    }

    conn = sqlite3.connect("/content/academia_portal.db")

    df = pd.read_sql_query("""
        SELECT
            o.opportunity_id,
            o.title,
            o.opportunity_type,
            o.location,
            c.company_name,
            s.skill_name
        FROM opportunities o
        JOIN companies c ON c.company_id = o.company_id
        LEFT JOIN opportunity_skills os
            ON os.opportunity_id = o.opportunity_id
        LEFT JOIN skills s ON s.skill_id = os.skill_id
        WHERE o.is_active = 1
    """, conn)

    conn.close()

    if df.empty:
        print("No active opportunities found.")
        return pd.DataFrame()

    results = []

    for opportunity_id, group in df.groupby("opportunity_id"):
        first = group.iloc[0]

        required = {
            normalize_skill(s)
            for s in group["skill_name"].dropna()
            if str(s).strip()
        }

        # Exclude opportunities with no mapped requirements
        if not required:
            continue

        matched = student_set & required
        missing = required - student_set
        coverage = len(matched) / len(required)

        results.append({
            "title": first["title"],
            "company": first["company_name"],
            "type": first["opportunity_type"],
            "location": first["location"],
            "match_score_percent": round(coverage * 100, 2),
            "matched_skills": ", ".join(sorted(matched)) or "None",
            "missing_skills": ", ".join(sorted(missing)) or "None",
            "requirements_available": True
        })

    if not results:
        print("No opportunities have mapped skill requirements.")
        return pd.DataFrame()

    return (
        pd.DataFrame(results)
        .sort_values("match_score_percent", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )


In [ ]:

student_skills = ["Python", "SQL", "Pandas"]

recommendations = recommend_from_database(
    student_skills,
    top_k=6
)

display(recommendations)


,title,company,type,location,match_score_percent,matched_skills,missing_skills,requirements_available
0,Data Analyst Intern,Example Analytics,internship,Remote,75.00,"pandas, python, sql",statistics,True
1,Junior Data Scientist,Example Research,job,Hybrid,60.00,"pandas, python, sql","machine learning, statistics",True
2,Deep Learning Intern,Example Vision,internship,Remote,33.33,python,"machine learning, tensorflow",True
3,Machine Learning Intern,Example AI,internship,Hybrid,25.00,python,"machine learning, scikit learn, statistics",True
4,BI Analyst,Example Consulting,job,On-site,25.00,sql,"communication, power bi, statistics",True
5,ML Engineer,Example Systems,job,Remote,25.00,python,"git, machine learning, tensorflow",True


In [ ]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("/content/academia_portal.db")
conn.execute("PRAGMA foreign_keys = ON")

# Demo student information
email = "student001@example.com"
full_name = "Demo Student"
degree = "M.Tech"
specialization = "Artificial Intelligence and Machine Learning"
graduation_year = 2027

student_skills = {
    "Python": 0.8,
    "SQL": 0.7,
    "Pandas": 0.8
}

# Create user if it does not exist
user = conn.execute(
    "SELECT user_id FROM users WHERE email = ?",
    (email,)
).fetchone()

if user:
    user_id = user[0]
else:
    user_id = conn.execute(
        """
        INSERT INTO users (email, password_hash, role)
        VALUES (?, ?, 'student')
        """,
        (email, "DEMO_ONLY_NOT_A_REAL_PASSWORD_HASH")
    ).lastrowid

# Create student profile if it does not exist
profile = conn.execute(
    "SELECT student_id FROM student_profiles WHERE user_id = ?",
    (user_id,)
).fetchone()

if profile:
    student_id = profile[0]
else:
    student_id = conn.execute(
        """
        INSERT INTO student_profiles
        (user_id, full_name, degree, specialization, graduation_year)
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            user_id, full_name, degree,
            specialization, graduation_year
        )
    ).lastrowid

# Save student skills
for skill_name, proficiency in student_skills.items():
    conn.execute(
        """
        INSERT OR IGNORE INTO skills (skill_name, category)
        VALUES (?, 'General')
        """,
        (skill_name,)
    )

    skill_id = conn.execute(
        "SELECT skill_id FROM skills WHERE LOWER(skill_name) = LOWER(?)",
        (skill_name,)
    ).fetchone()[0]

    conn.execute(
        """
        INSERT INTO student_skills
        (student_id, skill_id, proficiency, source)
        VALUES (?, ?, ?, 'self_reported')
        ON CONFLICT(student_id, skill_id)
        DO UPDATE SET proficiency = excluded.proficiency
        """,
        (student_id, skill_id, proficiency)
    )

conn.commit()

print("Student profile saved successfully!")
print("Student ID:", student_id)

display(pd.read_sql_query(
    """
    SELECT
        sp.student_id,
        sp.full_name,
        sp.degree,
        sp.specialization,
        sp.graduation_year,
        s.skill_name,
        ss.proficiency
    FROM student_profiles sp
    JOIN student_skills ss ON ss.student_id = sp.student_id
    JOIN skills s ON s.skill_id = ss.skill_id
    WHERE sp.student_id = ?
    ORDER BY s.skill_name
    """,
    conn,
    params=(student_id,)
))

conn.close()


Student profile saved successfully!
Student ID: 1


,student_id,full_name,degree,specialization,graduation_year,skill_name,proficiency
0,1,Demo Student,M.Tech,Artificial Intelligence and Machine Learning,2027,Pandas,0.8
1,1,Demo Student,M.Tech,Artificial Intelligence and Machine Learning,2027,Python,0.8
2,1,Demo Student,M.Tech,Artificial Intelligence and Machine Learning,2027,SQL,0.7


In [ ]:

import sqlite3
import pandas as pd

def recommend_for_student(student_id, top_k=5):
    conn = sqlite3.connect("/content/academia_portal.db")

    student_skills = pd.read_sql_query(
        """
        SELECT s.skill_name
        FROM student_skills ss
        JOIN skills s ON s.skill_id = ss.skill_id
        WHERE ss.student_id = ?
        """,
        conn,
        params=(student_id,)
    )["skill_name"].tolist()

    conn.close()

    if not student_skills:
        print("No skills found for this student.")
        return pd.DataFrame()

    print("Skills retrieved from database:", student_skills)

    return recommend_from_database(
        student_skills=student_skills,
        top_k=top_k
    )

recommendations = recommend_for_student(student_id=1, top_k=6)
display(recommendations)


Skills retrieved from database: ['Python', 'SQL', 'Pandas']


,title,company,type,location,match_score_percent,matched_skills,missing_skills,requirements_available
0,Data Analyst Intern,Example Analytics,internship,Remote,75.00,"pandas, python, sql",statistics,True
1,Junior Data Scientist,Example Research,job,Hybrid,60.00,"pandas, python, sql","machine learning, statistics",True
2,Deep Learning Intern,Example Vision,internship,Remote,33.33,python,"machine learning, tensorflow",True
3,Machine Learning Intern,Example AI,internship,Hybrid,25.00,python,"machine learning, scikit learn, statistics",True
4,BI Analyst,Example Consulting,job,On-site,25.00,sql,"communication, power bi, statistics",True
5,ML Engineer,Example Systems,job,Remote,25.00,python,"git, machine learning, tensorflow",True


In [ ]:

import pandas as pd
from pathlib import Path

data_dir = Path("/content/academia_industry_portal/data")
courses_df = pd.read_csv(data_dir / "courses.csv")

print("Available courses:")
display(courses_df)

def recommend_courses(missing_skills, courses, top_k=3):
    def normalize(value):
        return str(value).strip().lower().replace("-", " ")

    missing = {
        normalize(skill)
        for skill in missing_skills
        if str(skill).strip()
    }

    if not missing:
        print("No skill gaps identified. No courses needed.")
        return pd.DataFrame()

    results = []

    for _, course in courses.iterrows():
        taught = {
            normalize(skill)
            for skill in str(course["skills_taught"]).split(",")
            if skill.strip()
        }

        covered = missing & taught

        if covered:
            results.append({
                "course_name": course["course_name"],
                "provider": course["provider"],
                "skills_covered": ", ".join(sorted(covered)),
                "skills_covered_count": len(covered),
                "gap_coverage_percent": round(
                    len(covered) / len(missing) * 100, 2
                )
            })

    if not results:
        print("No available courses cover these skills.")
        return pd.DataFrame()

    return (
        pd.DataFrame(results)
        .sort_values(
            ["skills_covered_count", "course_name"],
            ascending=[False, True]
        )
        .head(top_k)
        .reset_index(drop=True)
    )

# Find courses for the top recommendation
if not recommendations.empty:
    best = recommendations.iloc[0]

    missing_skills = (
        []
        if best["missing_skills"] == "None"
        else best["missing_skills"].split(", ")
    )

    print("Target opportunity:", best["title"])
    print("Missing skills:", missing_skills)
    print("\nRecommended courses:")

    course_recommendations = recommend_courses(
        missing_skills,
        courses_df,
        top_k=3
    )

    display(course_recommendations)


Available courses:


,course_id,course_name,skills_taught,level,provider
0,C001,Python for Data Analysis,"Python, Pandas, SQL",Beginner,University
1,C002,Machine Learning Fundamentals,"Python, Machine Learning, Statistics, Scikit-l...",Intermediate,University
2,C003,Deep Learning with TensorFlow,"Python, TensorFlow, Machine Learning",Advanced,University
3,C004,Business Intelligence,"SQL, Power BI, Statistics",Intermediate,Industry Partner


Target opportunity: Data Analyst Intern
Missing skills: ['statistics']

Recommended courses:


,course_name,provider,skills_covered,skills_covered_count,gap_coverage_percent
0,Business Intelligence,Industry Partner,statistics,1,100.0
1,Machine Learning Fundamentals,University,statistics,1,100.0


In [ ]:
%pip install -q streamlit

In [ ]:

from pathlib import Path

project_dir = Path("/content/academia_industry_portal")
project_dir.mkdir(parents=True, exist_ok=True)

app_code = r'''
import sqlite3
from pathlib import Path

import pandas as pd
import streamlit as st

DB_PATH = Path("/content/academia_portal.db")
DATA_DIR = Path("/content/academia_industry_portal/data")

st.set_page_config(
    page_title="Academia–Industry Portal",
    page_icon="🎓",
    layout="wide"
)

st.title("🎓 Academia–Industry Collaboration Portal")
st.caption("Skill mapping • Opportunity matching • Learning recommendations")

def load_data():
    if not DB_PATH.exists():
        return pd.DataFrame(), pd.DataFrame()

    with sqlite3.connect(DB_PATH) as conn:
        opportunities = pd.read_sql_query("""
            SELECT
                o.opportunity_id,
                o.title,
                o.opportunity_type,
                o.location,
                c.company_name,
                s.skill_name
            FROM opportunities o
            JOIN companies c ON c.company_id = o.company_id
            LEFT JOIN opportunity_skills os
                ON os.opportunity_id = o.opportunity_id
            LEFT JOIN skills s ON s.skill_id = os.skill_id
            WHERE o.is_active = 1
        """, conn)

        skills = pd.read_sql_query(
            "SELECT skill_name FROM skills ORDER BY skill_name",
            conn
        )

    return opportunities, skills

def normalize(skill):
    return str(skill).strip().lower().replace("-", " ")

def match_opportunities(student_skills, data):
    results = []

    for opportunity_id, group in data.groupby("opportunity_id"):
        first = group.iloc[0]

        required = {
            normalize(s)
            for s in group["skill_name"].dropna()
            if str(s).strip()
        }

        if not required:
            continue

        matched = required & student_skills
        missing = required - student_skills

        results.append({
            "Opportunity": first["title"],
            "Company": first["company_name"],
            "Type": first["opportunity_type"].title(),
            "Location": first["location"],
            "Match (%)": round(len(matched) / len(required) * 100, 1),
            "Matched skills": ", ".join(sorted(matched)) or "None",
            "Missing skills": ", ".join(sorted(missing)) or "None"
        })

    if not results:
        return pd.DataFrame()

    return pd.DataFrame(results).sort_values(
        "Match (%)", ascending=False
    ).reset_index(drop=True)

st.sidebar.header("Student Profile")

opportunities, skills_catalogue = load_data()

if opportunities.empty:
    st.error("No active opportunities found. Check the database.")
    st.stop()

available_skills = skills_catalogue["skill_name"].dropna().tolist()

default_skills = [
    s for s in ["Python", "SQL", "Pandas"]
    if s in available_skills
]

selected_skills = st.sidebar.multiselect(
    "Select your skills",
    options=available_skills,
    default=default_skills
)

if not selected_skills:
    st.info("Select at least one skill from the sidebar.")
    st.stop()

student_skills = {normalize(s) for s in selected_skills}
results = match_opportunities(student_skills, opportunities)

total_opportunities = opportunities["opportunity_id"].nunique()
matched_count = len(results)

col1, col2, col3 = st.columns(3)
col1.metric("Active opportunities", total_opportunities)
col2.metric("Skills selected", len(selected_skills))
col3.metric("Recommendations", matched_count)

st.subheader("Recommended Jobs and Internships")

if results.empty:
    st.warning("No opportunities have mapped skill requirements.")
else:
    st.dataframe(results, use_container_width=True, hide_index=True)

    best = results.iloc[0]
    st.subheader("Skill-gap learning recommendations")

    missing_skills = (
        [] if best["Missing skills"] == "None"
        else best["Missing skills"].split(", ")
    )

    if not missing_skills:
        st.success(
            f"No listed skill gaps for {best['Opportunity']}."
        )
    else:
        st.write(
            f"**Target:** {best['Opportunity']} — "
            f"missing skills: {', '.join(missing_skills)}"
        )

        course_path = DATA_DIR / "courses.csv"

        if course_path.exists():
            courses = pd.read_csv(course_path)
            course_results = []

            for _, course in courses.iterrows():
                taught = {
                    normalize(s)
                    for s in str(course["skills_taught"]).split(",")
                    if s.strip()
                }
                covered = student_skills.union()  # Empty set for clarity
                covered = set(missing_skills) & taught

                if covered:
                    course_results.append({
                        "Course": course["course_name"],
                        "Provider": course["provider"],
                        "Skills covered": ", ".join(sorted(covered))
                    })

            if course_results:
                st.dataframe(
                    pd.DataFrame(course_results),
                    use_container_width=True,
                    hide_index=True
                )
            else:
                st.info(
                    "No listed courses cover these missing skills."
                )
        else:
            st.warning("courses.csv was not found.")

st.caption(
    "Match percentage represents required-skill coverage, "
    "not the probability of getting hired. Demonstration dataset only."
)
'''

app_path = project_dir / "app.py"
app_path.write_text(app_code, encoding="utf-8")

print("Dashboard file created:", app_path)
print("Next, start Streamlit to launch the dashboard.")


Dashboard file created: /content/academia_industry_portal/app.py
Next, start Streamlit to launch the dashboard.


In [ ]:
!cat /content/streamlit.log

cat: /content/streamlit.log: No such file or directory


In [ ]:
!ps aux | grep '[s]treamlit'
!curl -I http://localhost:8501

curl: (7) Failed to connect to localhost port 8501 after 0 ms: Couldn't connect to server


In [ ]:
!pkill -f streamlit

In [ ]:
!streamlit run /content/academia_industry_portal/app.py \
  --server.port 8502 \
  --server.headless true \
  > /content/streamlit.log 2>&1 &

In [ ]:
!cat /content/streamlit.log

In [ ]:
!curl -I http://localhost:8502

curl: (7) Failed to connect to localhost port 8502 after 0 ms: Couldn't connect to server


In [ ]:
!curl -s -o /dev/null -w "HTTP status: %{http_code}\n" http://127.0.0.1:8502

HTTP status: 000


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack /tmp/cloudflared.deb ...
Unpacking cloudflared (2026.10.7) ...
Setting up cloudflared (2026.10.7) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
!cloudflared tunnel --url http://127.0.0.1:8502 --no-autoupdate

2026-10-31T10:24:53Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-31T10:24:53Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-31T10:24:56Z INF +--------------------------------------------------------------------------------------------+
2026-10-31T10:24:56Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-31T10:24:56Z INF |  https://existence-distributed-golden-judges.trycloudf

In [ ]:

%%writefile /content/academia_industry_portal/app.py
import sqlite3
from pathlib import Path

import pandas as pd
import streamlit as st

DB_PATH = Path("/content/academia_portal.db")
COURSES_PATH = Path("/content/academia_industry_portal/data/courses.csv")

st.set_page_config(
    page_title="Academia–Industry Portal",
    page_icon="🎓",
    layout="wide",
)

st.title("🎓 Academia–Industry Collaboration Portal")
st.caption("Student skill mapping • Job matching • Internship recommendations")

def normalize(skill):
    value = str(skill).strip().lower().replace("-", " ")
    value = " ".join(value.split())
    aliases = {
        "ml": "machine learning",
        "sklearn": "scikit learn",
        "scikit learn": "scikit learn",
        "powerbi": "power bi",
    }
    return aliases.get(value, value)

def load_data():
    with sqlite3.connect(DB_PATH) as conn:
        opportunities = pd.read_sql_query("""
            SELECT
                o.opportunity_id,
                o.title,
                o.opportunity_type,
                o.location,
                c.company_name,
                s.skill_name
            FROM opportunities AS o
            JOIN companies AS c
                ON c.company_id = o.company_id
            LEFT JOIN opportunity_skills AS os
                ON os.opportunity_id = o.opportunity_id
            LEFT JOIN skills AS s
                ON s.skill_id = os.skill_id
            WHERE o.is_active = 1
        """, conn)

        skills = pd.read_sql_query("""
            SELECT skill_name
            FROM skills
            ORDER BY skill_name
        """, conn)

    return opportunities, skills

def recommend_opportunities(student_skills, data):
    results = []

    for _, group in data.groupby("opportunity_id"):
        first = group.iloc[0]
        required = {
            normalize(skill)
            for skill in group["skill_name"].dropna()
            if str(skill).strip()
        }

        if not required:
            continue

        matched = required & student_skills
        missing = required - student_skills

        results.append({
            "Opportunity": first["title"],
            "Company": first["company_name"],
            "Type": str(first["opportunity_type"]).title(),
            "Location": first["location"],
            "Match (%)": round(100 * len(matched) / len(required), 1),
            "Matched skills": ", ".join(sorted(matched)) or "None",
            "Missing skills": ", ".join(sorted(missing)) or "None",
        })

    if not results:
        return pd.DataFrame()

    return pd.DataFrame(results).sort_values(
        "Match (%)", ascending=False
    ).reset_index(drop=True)

if not DB_PATH.exists():
    st.error(f"Database not found: {DB_PATH}")
    st.stop()

try:
    opportunities, skill_catalogue = load_data()
except Exception as error:
    st.error(f"Could not load database tables: {error}")
    st.stop()

if opportunities.empty:
    st.warning("No active opportunities found in the database.")
    st.stop()

available_skills = skill_catalogue["skill_name"].dropna().tolist()
defaults = [s for s in ["Python", "SQL", "Pandas"] if s in available_skills]

st.sidebar.header("Student Profile")
selected_skills = st.sidebar.multiselect(
    "Select your skills",
    options=available_skills,
    default=defaults,
)

if not selected_skills:
    st.info("Choose at least one skill in the sidebar.")
    st.stop()

student_skills = {normalize(s) for s in selected_skills}
results = recommend_opportunities(student_skills, opportunities)

col1, col2, col3 = st.columns(3)
col1.metric("Active opportunities", opportunities["opportunity_id"].nunique())
col2.metric("Skills selected", len(selected_skills))
col3.metric("Opportunities matched", len(results))

st.subheader("Recommended jobs and internships")
st.dataframe(results, use_container_width=True, hide_index=True)

best = results.iloc[0]
missing_skills = (
    [] if best["Missing skills"] == "None"
    else best["Missing skills"].split(", ")
)

st.subheader("Learning recommendations")
st.write(f"**Top match:** {best['Opportunity']} at {best['Company']}")

if not missing_skills:
    st.success("You cover all listed skills for this opportunity.")
elif not COURSES_PATH.exists():
    st.warning("courses.csv was not found.")
else:
    courses = pd.read_csv(COURSES_PATH)
    course_results = []

    for _, course in courses.iterrows():
        taught = {
            normalize(skill)
            for skill in str(course.get("skills_taught", "")).split(",")
            if skill.strip()
        }
        covered = set(missing_skills) & taught

        if covered:
            course_results.append({
                "Course": course.get("course_name", "Unnamed course"),
                "Provider": course.get("provider", "Not specified"),
                "Skills covered": ", ".join(sorted(covered)),
            })

    if course_results:
        st.dataframe(
            pd.DataFrame(course_results),
            use_container_width=True,
            hide_index=True,
        )
    else:
        st.info("No courses in the current catalogue cover these missing skills.")

st.caption(
    "Match percentage measures required-skill coverage, not the probability "
    "of getting hired. This is a demonstration dataset."
)


In [ ]:
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

app_path = "/content/academia_industry_portal/app.py"
log_path = "/content/streamlit_8503.log"

log_file = open(log_path, "w")

process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", app_path,
        "--server.port", "8503",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

for attempt in range(20):
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:8503", timeout=2
        ) as response:
            print("Dashboard status:", response.status)
            print("Streamlit is running on port 8503.")
            break
    except Exception:
        time.sleep(1)
else:
    print(Path(log_path).read_text()[-5000:])

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:8503 --no-autoupdate

2026-10-31T11:32:15Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-31T11:32:15Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-31T11:32:19Z INF +--------------------------------------------------------------------------------------------+
2026-10-31T11:32:19Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-31T11:32:19Z INF |  https://bucks-reporter-hobby-typical.trycloudflare.co